[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aloshdenny/lbs-agentic-ai/blob/main/notebooks/handoffs-and-tools.ipynb)

# Two Agents, Two Tools, One Handoff

From Chapter 02 and 03: no framework, just the Groq API by hand. One cell to install, a few reusable cells (client, tools, agents), then the orchestration loop that ties it together.

Two agents share this notebook:

- **Generalist Agent** — everyday questions, can browse the web
- **Specialist Agent** — numeric computation, can run code

Both agents get both tools (`browser_search`, `code_interpreter`). The only thing that differs is *when each one decides to hand off* to the other.

In [ ]:
%pip install -q groq

## Reusable: the client

One client, defined once. Every agent below calls through this same object — swap `MODEL` here if you want to try a different one.

In [ ]:
from groq import Groq
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "PASTE_YOUR_GROQ_API_KEY_HERE")
MODEL = "openai/gpt-oss-20b"

client = Groq(api_key=GROQ_API_KEY)

## Reusable: the tools

`browser_search` and `code_interpreter` are Groq's own built-in tools — Groq runs them server-side, so the code never has to implement them.

The two `transfer_to_*` tools are ours: plain function-calling definitions with no implementation behind them. When the model calls one, that *is* the handoff signal — our code below is what actually acts on it.

One quirk worth knowing: once a transfer tool call shows up anywhere in the conversation, Groq expects it to stay declared in `tools` on every later request in that same exchange, even for the agent that didn't call it. Simplest fix: always pass both transfer tools, and let each agent's system prompt tell it not to call the one that would just transfer to itself.

In [ ]:
BASE_TOOLS = [{"type": "browser_search"}, {"type": "code_interpreter"}]

TRANSFER_TO_SPECIALIST = {
    "type": "function",
    "function": {
        "name": "transfer_to_specialist",
        "description": "Hand off to the Specialist Agent for detailed numeric computation, data analysis, or multi-step calculation.",
        "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]},
    },
}
TRANSFER_TO_GENERALIST = {
    "type": "function",
    "function": {
        "name": "transfer_to_generalist",
        "description": "Hand off to the Generalist Agent for everyday questions, lookups, or anything not requiring deep computation.",
        "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]},
    },
}

ALL_TOOLS = BASE_TOOLS + [TRANSFER_TO_SPECIALIST, TRANSFER_TO_GENERALIST]

## Reusable: the two agents

Just a name and a system prompt each. Everything else (the client, the tools) is shared.

In [ ]:
AGENTS = {
    "generalist": {
        "name": "Generalist Agent",
        "system_prompt": (
            "You are the Generalist Agent. You answer everyday questions and can browse the web for facts. "
            "If the user asks for a detailed numeric computation, statistics, or multi-step calculation, "
            "call transfer_to_specialist instead of answering yourself. Never call transfer_to_generalist, "
            "that would just be transferring to yourself."
        ),
    },
    "specialist": {
        "name": "Specialist Agent",
        "system_prompt": (
            "You are the Specialist Agent. You handle numeric computation and data analysis, using the code "
            "interpreter tool for any real calculation. If the user asks a general knowledge or everyday "
            "question instead, call transfer_to_generalist. Never call transfer_to_specialist, that would "
            "just be transferring to yourself."
        ),
    },
}

## The orchestration

One function, one turn: ask the active agent, and if it calls a `transfer_to_*` tool, switch the active agent and ask again, up to a small hop limit. The handoff plumbing (the tool call and its response) stays local to this function — only the clean user/assistant text joins the long-lived conversation, so old handoffs never come back to cause tool-declaration mismatches on a later turn.

In [ ]:
def run_turn(messages, active_key):
    working = list(messages)  # scratch copy — handoff plumbing never joins long-term memory
    hops = 0
    while hops < 3:
        agent = AGENTS[active_key]
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": agent["system_prompt"]}] + working,
            tools=ALL_TOOLS,
            temperature=0.3,
            max_completion_tokens=1024,
        )
        msg = response.choices[0].message
        transfer_call = next(
            (tc for tc in (msg.tool_calls or []) if tc.function.name.startswith("transfer_to_")), None
        )
        if transfer_call:
            working.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [{
                    "id": transfer_call.id,
                    "type": "function",
                    "function": {"name": transfer_call.function.name, "arguments": transfer_call.function.arguments},
                }],
            })
            working.append({"role": "tool", "tool_call_id": transfer_call.id, "content": "Handed off."})
            active_key = transfer_call.function.name.replace("transfer_to_", "")
            print(f"  [handing off to {AGENTS[active_key]['name']}]")
            hops += 1
            continue
        messages.append({"role": "assistant", "content": msg.content})
        print(f"{agent['name']}: {msg.content}")
        return messages, active_key
    messages.append({"role": "assistant", "content": "(gave up after too many handoffs)"})
    return messages, active_key

## Try it

Starts with the Generalist Agent. Ask it something that needs real computation partway through, and watch it hand off.

In [ ]:
messages = []
active_key = "generalist"

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("exit", "quit"):
        break
    messages.append({"role": "user", "content": user_input})
    messages, active_key = run_turn(messages, active_key)